# 🧠 ClipFactory AI Director — Phase 5: SaaS Stabilization
> The ultimate AI-powered video repurposing engine.

**Runtime:** Make sure you're on **T4 GPU** (Runtime > Change runtime type > T4 GPU).

### Installation Notes
- **Cell 1 (Setup):** Installs all dependencies (~3-4 min). We use prebuilt wheels for LLaMA to skip the long compile time.
- **Building UI:** The dashboard build takes ~2 minutes. Please don't refresh the page while it's running.


In [ ]:
# ─── CELL 0: Optional Google Drive Mount ──────────────────────────────────────────
import os
if os.path.exists('/content/drive/MyDrive'):
    print('✅ Drive successfully detected.')
else:
    print('⚠️ Drive not mounted. Files will be saved to temporary storage.')

In [ ]:
# ─── CELL 1: Setup & Installation (Run once per session) ───────────────────────────
import os
REPO_URL = 'https://github.com/NiL4gh/clip-factory.git'
REPO_DIR = '/content/clip-factory'

# 1. Clone or Update the repository
if not os.path.exists(REPO_DIR):
    print(f'🚀 Cloning {REPO_URL}...')
    !git clone {REPO_URL} {REPO_DIR}
else:
    print('🔄 Repository exists. Pulling latest changes...')
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)

# 2. Install Python dependencies
print('🐍 Installing Python dependencies (~2 min)...')
!pip install -r requirements.txt --quiet
!pip install pyngrok --quiet

print('🧠 Installing LLaMA-CPP (using prebuilt CUDA wheels)...')
!pip install llama-cpp-python --prefer-binary --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124

# 3. Build Frontend (Next.js Static Export)
print('⚛️  Installing Node dependencies (~1-2 min)...')
frontend_path = os.path.join(REPO_DIR, 'frontend')
if not os.path.exists(os.path.join(frontend_path, 'node_modules')):
    !cd {frontend_path} && npm install --no-audit --no-fund --quiet

print('🏗️  Building production dashboard (optimizing assets)...')
!cd {frontend_path} && NEXT_TELEMETRY_DISABLED=1 npm run build

print('\n✅ Setup complete! All dependencies installed and Dashboard built.')
print('🚀 Now run Cell 2 to launch the dashboard.')

In [ ]:
# ─── CELL 2: Launch ClipFactory.ai ────────────────────────────────────────────────
import os, sys, subprocess, threading, time, socket

REPO_DIR = '/content/clip-factory'
if not os.path.exists(REPO_DIR):
    print('❌ ERROR: Repository not found! Please run Cell 1 first.')
else:
    # Kill stale processes on port 8000
    subprocess.run('fuser -k 8000/tcp 2>/dev/null || true', shell=True, stdout=subprocess.DEVNULL)
    time.sleep(1)

    # Check for GPU
    try:
        subprocess.check_output('nvidia-smi')
        print('✅ GPU Detected. LLaMA 3 will run with acceleration.')
    except:
        print('❌ WARNING: No GPU detected! The AI Director will be slow.')

    os.chdir(REPO_DIR)
    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)

    # Auto-detect Drive for persistence
    if os.path.exists('/content/drive/MyDrive'):
        BASE = '/content/drive/MyDrive/clip_factory'
        print('✅ Saving models and projects to Google Drive.')
    else:
        BASE = '/content/clip_factory'
        print('⚠️ Using local ephemeral storage.')

    # ── Start FastAPI backend (serves API + static frontend) ──────────────────
    def run_backend():
        import uvicorn
        uvicorn.run('server.main:app', host='127.0.0.1', port=8000, log_level='info')

    print('🚀 Starting FastAPI backend...')
    backend_thread = threading.Thread(target=run_backend, daemon=True)
    backend_thread.start()

    # Wait for the backend to be ready
    print('⏳ Waiting for port 8000 to be active...')
    ready = False
    for _ in range(30):
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            if s.connect_ex(('127.0.0.1', 8000)) == 0:
                ready = True
                break
        time.sleep(1)
    
    if not ready:
        print('❌ ERROR: Backend failed to start on port 8000.')
    else:
        print('✅ FastAPI backend is LIVE.')
        
        # ── Create single ngrok tunnel ──────────────────────────────────────────
        from pyngrok import ngrok
        try:
            ngrok_token = '3DHR9cBJA1DwgIYWqm28CFKjkQT_3E1sWDqq5LvfsSYdQzkUm'
            ngrok.set_auth_token(ngrok_token)
            tunnel = ngrok.connect('127.0.0.1:8000', 'http')
            public_url = tunnel.public_url

            # Inject API URL into the static build
            out_dir = os.path.join(REPO_DIR, 'frontend', 'out')
            if os.path.isdir(out_dir):
                config_js = os.path.join(out_dir, '_config.js')
                with open(config_js, 'w') as f:
                    f.write(f'window.__NEXT_PUBLIC_API_URL = "{public_url}/api";')
                idx_html = os.path.join(out_dir, 'index.html')
                if os.path.exists(idx_html):
                    with open(idx_html, 'r') as f:
                        html = f.read()
                    if '_config.js' not in html:
                        html = html.replace('<head>', '<head><script src="/_config.js"></script>')
                        with open(idx_html, 'w') as f:
                            f.write(html)
                print('✅ API URL auto-injected into dashboard.')

            print(f'\n' + '═' * 60)
            print(f'  🚀  ClipFactory.ai is LIVE')
            print(f'  Dashboard:  {public_url}')
            print(f'  API:        {public_url}/api')
            print('═' * 60)
            print(f'\nOpen the Dashboard URL above — everything is ready to go.')

            # Keep cell alive
            while True: time.sleep(60)
        except KeyboardInterrupt:
            ngrok.kill()
            print('Tunnel closed.')
